# DACON: CPU ADS and A100 PANNs experiments

Portable launcher for reviewed repository CLIs. It does not install packages, download data, create a workspace, or submit predictions. Select exactly one mode with an absolute JSON file in `DACON_EXPERIMENT_CONFIG` before starting the kernel. Run all cells in order in an already prepared environment.

- `cpu_ads`: eligible cached features, optional preserved-head reference and independent/shared/joint head diagnostics. The CLI validates feature provenance and roles; existing output is refused.
- `a100_panns`: existing A100 only, pre-uploaded release only. The launcher verifies archive/data identities, runs CUDA parity, resumes caches/checkpoints, and records its bounded exit. The external controller owns workspace pause and state verification. Keep the original absolute deadline on every resume. Run this mode only under the externally supervised VESSL startup for the exact existing workspace; that supervisor must pause on prelaunch errors or expiry. That startup supplies inline `PANNS_CONTROLLER_CONTRACT`; the guard verifies its exact workspace, original deadline and uploaded bundle identity. This notebook is not a standalone workspace cost controller.
- `cpu_panns_report`: regenerate the recorded three-seed comparison from a complete run on CPU.

**No execution or quality result is embedded here.** Local CPU runs must execute the code cells inside the project's approved offline sandbox. If that sandbox blocks Jupyter sockets, run the unchanged code cells in one Python namespace and record that Jupyter transport/UI were not validated. CUDA is hidden from CPU children. A100 automatic pause may disconnect the kernel; retrieve the persisted CLI receipts and result JSON to establish completion. A successful cell alone is not a leaderboard improvement.

Common JSON keys: `mode`, `code_dir` (absolute directory containing the named CLI), `log` (new absolute file), optional `python` (existing CPU interpreter). Mode keys are listed in the first code cell. CPU ADS also accepts optional `reference`. Choose either current `manifest`/`feature_manifest`/`cache` inputs or the explicit `legacy_cache_root`/`legacy_bank_dir`/`legacy_producer` inputs; do not mix these contracts. The CLI records the selected cohort, exclusions and common epoch budget. A100 `code_dir` is the versioned release directory containing launcher, guard, lock and bundle. Do not put credentials in the configuration.


In [ ]:
import collections, hashlib, json, math, os, signal, subprocess, sys, time
from pathlib import Path

def require(condition: bool, message: str) -> None:
    if not condition:
        raise ValueError(message)

def digest(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

config_path = Path(os.environ["DACON_EXPERIMENT_CONFIG"])
require(config_path.is_absolute(), "DACON_EXPERIMENT_CONFIG must be absolute")
config = json.loads(config_path.read_text())
config_hash = digest(config_path)
mode = config["mode"]
options = {
    "cpu_ads": {"manifest", "feature_manifest", "cache", "lineage", "output", "minutes", "epochs", "reference",
                "legacy_cache_root", "legacy_bank_dir", "legacy_producer"},
    "cpu_panns_report": {"run"},
    "a100_panns": {"shared", "work", "deadline_state", "deadline_epoch", "hours", "training_hours"},
}
require(mode in options, "Choose cpu_ads, cpu_panns_report, or a100_panns")
require(not set(config) - options[mode] - {"mode", "code_dir", "python", "log"}, "Unknown config key")
def path(key: str, exists: bool = True) -> Path:
    value = Path(config[key])
    require(value.is_absolute(), key + " must be absolute")
    require(not exists or value.exists(), key + " is missing")
    return value.resolve()
code_dir, log_path = path("code_dir"), path("log", False)
require(log_path.parent.is_dir() and not log_path.exists(), "Use a new log in an existing directory")
env = {key: value for key, value in os.environ.items() if not key.startswith("PANNS_") or key == "PANNS_CONTROLLER_CONTRACT"}
env.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", PYTHONDONTWRITEBYTECODE="1")
python = config.get("python", sys.executable)
require(Path(python).is_absolute() and Path(python).is_file(), "Use an existing absolute interpreter path")


In [ ]:
if mode.startswith("cpu_"):
    env["CUDA_VISIBLE_DEVICES"] = ""
    env.update(OMP_NUM_THREADS="4", MKL_NUM_THREADS="4")
if mode == "cpu_ads":
    minutes, epochs = float(config["minutes"]), config["epochs"]
    require(type(epochs) is int, "epochs must be an integer")
    require(0 < minutes <= 30 and 1 <= epochs <= 20, "CPU bounds: <=30 minutes and <=20 epochs")
    legacy_keys = {"legacy_cache_root", "legacy_bank_dir", "legacy_producer"}
    modern_keys = {"manifest", "feature_manifest", "cache"}
    legacy = bool(legacy_keys & config.keys())
    require(not legacy or (legacy_keys <= config.keys() and not modern_keys & config.keys()),
            "Choose a complete legacy input contract or current feature-manifest/cache contract")
    inputs = ["legacy_cache_root", "legacy_bank_dir", "legacy_producer", "lineage"] if legacy else [
        "manifest", "feature_manifest", "cache", "lineage"]
    command = [python, str(code_dir / "ads_cached_experiments.py")]
    for key in inputs:
        command += ["--" + key.replace("_", "-"), str(path(key))]
    output = path("output", False)
    require(not output.exists(), "CPU ADS output must be new; partial runs are not silently resumed")
    command += ["--output", str(output), "--minutes", str(minutes), "--epochs", str(epochs)]
    if "reference" in config:
        command += ["--reference", str(path("reference"))]
    report_path, timeout = output / "summary.json", minutes * 60 + 60
elif mode == "cpu_panns_report":
    run = path("run")
    command = [python, str(code_dir / "panns_cloud_report.py"), "--run", str(run)]
    report_path = Path(json.loads(run.read_text())["output"]) / "development_report.json"
    timeout = 1800
else:
    require("python" not in config, "A100 uses the launcher-verified Python environment")
    require(bool(env.get("PANNS_CONTROLLER_CONTRACT")), "A100 requires the external startup controller contract")
    deadline, hours, training = (float(config[k]) for k in ("deadline_epoch", "hours", "training_hours"))
    require(all(map(math.isfinite, (deadline, hours, training))), "Finite allowances required")
    require(0 < deadline - time.time() <= hours * 3600 <= 72 * 3600 and 0 < training <= hours,
            "Retain original future deadline; 0 < training_hours <= hours <= 72")
    require(sys.platform.startswith("linux"), "A100 mode requires the existing Linux A100 workspace")
    device = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                            check=True, capture_output=True, text=True, timeout=10).stdout.splitlines()
    require(len(device) == 1 and "A100" in device[0], "Expected exactly one existing A100")
    shared, work, state = path("shared"), path("work", False), path("deadline_state", False)
    require(code_dir.is_relative_to(shared), "A100 code_dir must be the pre-uploaded release under shared")
    for name in ("launch_lock.json", "run_bundle.tar", "panns_launch_guard.py"):
        require((code_dir / name).is_file(), "Incomplete pre-uploaded release: " + name)
    env.update(PANNS_SHARED=str(shared), PANNS_WORK=str(work), PANNS_DEADLINE_STATE=str(state),
               PANNS_DEADLINE_EPOCH=str(deadline))
    command = ["/bin/bash", str(code_dir / "panns_a100_launch.sh"), "--hours", str(hours),
               "--training-hours", str(training)]
    report_path, timeout = None, deadline - time.time() + 10
entry = Path(command[1])
require(entry.is_file(), "Experiment entry point is missing")
entry_hash = digest(entry)
print(json.dumps({"mode": mode, "config_sha256": config_hash, "entry_sha256": entry_hash,
                  "argv": command, "log": str(log_path)}, indent=2))


## Execution and resume

The complete command and source/config hashes appear before execution. Logs stream to disk; only the final 20 lines are shown. The execution cell refuses a changed config or entry point. After any interruption, restart the kernel and rerun all cells with a new log path. CPU ADS needs a new output directory; A100 retains its original work directory and deadline state. The existing CLI owns all split, cache, optimizer and scoring rules. A100 validation before the launcher starts does not pause the workspace; the external startup supervisor covers that interval.


In [ ]:
require(digest(config_path) == config_hash and digest(entry) == entry_hash,
        "Config or entry point changed; restart the kernel and run all cells")
print("Running; full stdout/stderr streams to", log_path, flush=True)
with log_path.open("x") as log:
    process = subprocess.Popen(command, cwd=code_dir, env=env, stdout=log,
                               stderr=subprocess.STDOUT, start_new_session=True)
    try:
        returncode = process.wait(timeout=timeout)
    except (KeyboardInterrupt, subprocess.TimeoutExpired):
        process.send_signal(signal.SIGTERM)
        try:
            process.wait(timeout=100)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait(timeout=5)
        raise
with log_path.open() as log:
    print("".join(collections.deque(log, maxlen=20)))
require(returncode == 0, "Experiment failed; inspect the saved log before resuming")


## Recorded result

This cell reads the CLI's saved report only. Inspect the linked report for cohort denominators, per-condition metrics, incomplete arms, uncertainty and provenance before comparing models. A CPU diagnostic is not independent evaluation; no mode automatically adopts a model or submits to DACON.


In [ ]:
if mode == "a100_panns":
    run = json.loads((work / "run.json").read_text())
    report_path = Path(run["output"]) / "pipeline_complete.json"
require(report_path is not None and report_path.is_file(), "CLI exited without its required completion report")
record = json.loads(report_path.read_text())
keys = ("complete", "seconds", "peak_rss_gib", "selected_candidate", "independent_evaluation",
        "adoption_authorized", "quality_evidence", "winner", "deadline_epoch")
print(json.dumps({"report": str(report_path), "report_sha256": digest(report_path),
                  "recorded_values": {key: record[key] for key in keys if key in record}}, indent=2))
if mode == "cpu_ads":
    require(record["complete"], "CPU deadline censored the experiment; this is not a complete comparison")
